# Week 6: Validation Audit — Honest Model Review

This notebook is based on the FlyRank SEO research paper and the same careful review lens we apply to our own Week 5 model.

The paper is useful because it shows two lessons clearly:
- position is a very strong signal for CTR, and
- honest validation matters. A random split can make a model look stronger than it is.

This notebook applies that same rigor to my own model. The goal is to ask better methodology questions, not to score the paper or to overstate certainty.

## 1) Two paper findings + my methodology questions

I am reading the paper as a reviewer, not as a defender. In that spirit, I am focusing on two findings that matter most for trust in the result.

### Finding 1: Position remains the dominant signal for CTR

**Paper claim:** The paper reports that search position is by far the strongest observable driver of CTR, and that the effect is much larger than many other descriptive features.

**My methodology question:** If position is so dominant, are we measuring actual content quality or just how much search exposure the page already has? Does the model treat a page in position 1 differently from a page in position 10 purely because of exposure, not because of content quality or metadata?

**Why I ask this:** This matters because a result can look predictive without proving that the content itself is the cause. Position is a strong confounder. A model can look smart if it is mostly learning the SERP context instead of the underlying content opportunity.

**What this means for my own work:** I should not claim that a low-CTR page is necessarily a content problem without also checking position, visibility, and demand. In other words, the paper makes me more careful about interpretation, not more cynical about the data.

### Finding 2: Honest validation changes the story

**Paper claim:** The paper says that model performance appears stronger under easier validation and weaker under more honest validation, especially when the split is grouped or otherwise respects client structure. In plain terms: the winning model may look much better before validation is made more realistic.

**My methodology question:** Did the evaluation design allow the same client patterns, page families, or time effects to leak across train and test? If so, what happens when the model is tested on unseen clients or a more realistic split?

**Why I ask this:** This is the main lesson for risk management in ML. Overstated results are not necessarily wrong in a technical sense; they are just often not the right answer to the business question. A model that only works nearby the same patterns can still look strong in a weak split.

**What this means for my own work:** My Week 5 model should be re-run under a grouped or time-aware split and compared directly against the earlier random-split result. The honest split is not just a stricter test — it is the version most relevant to decision-support.

## 2) My model under an honest split (before/after)

This section tests the same question the paper raises: does my model still look useful when I validate it honestly?

I will compare:
- a simpler random row-level split (the kind of split that can look optimistic); and
- a grouped client-holdout split (more honest and more realistic).

The comparison helps answer whether my pattern is real or whether it mainly reflects repeated client or page patterns.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

# Repo setup
root = Path.cwd()
if not (root / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            root = candidate
            break

sys.path.insert(0, str(root / 'scripts'))
sys.path.insert(0, str(root))

feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = root / 'data' / 'processed' / 'baseline_refresh_queue.csv'

if not feature_path.exists():
    subprocess.run(['python', str(root / 'scripts' / '01_prepare_features.py')], cwd=root, check=True)

if not baseline_path.exists():
    subprocess.run(['python', str(root / 'scripts' / '02_baseline_score.py')], cwd=root, check=True)

print(f'Repo root: {root}')
print('Data pipeline ready.')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

frame = pd.read_csv(feature_path)
baseline = pd.read_csv(baseline_path)

numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]

numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = frame[categorical_features].fillna('unknown').astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
y = frame['is_declining_label'].astype(int)

print(f'Rows: {len(frame):,}')
print(f'Feature matrix shape: {X.shape}')
print(f'Positive rate: {y.mean():.3f}')

In [ ]:
# BEFORE: random row-level split
print('=== BEFORE: random split ===')
rng = np.random.default_rng(42)
client_values = frame['client_id'].fillna('unknown').astype(str).to_numpy()
all_clients = np.unique(client_values)
shuffled_clients = rng.permutation(all_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = frame['client_id'].fillna('unknown').astype(str).isin(test_clients).to_numpy()
train_idx_before = np.arange(len(frame))[~test_mask]
test_idx_before = np.arange(len(frame))[test_mask]

if len(train_idx_before) == 0 or len(test_idx_before) == 0 or y.iloc[train_idx_before].nunique() < 2 or y.iloc[test_idx_before].nunique() < 2:
    train_idx_before, test_idx_before = train_test_split(np.arange(len(frame)), test_size=0.2, random_state=42, stratify=y)

X_train_before = X.iloc[train_idx_before]
X_test_before = X.iloc[test_idx_before]
y_train_before = y.iloc[train_idx_before]
y_test_before = y.iloc[test_idx_before]

model_before = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])
model_before.fit(X_train_before, y_train_before)
prob_before = model_before.predict_proba(X_test_before)[:, 1]
pred_before = (prob_before >= 0.5).astype(int)

metrics_before = {
    'accuracy': accuracy_score(y_test_before, pred_before),
    'precision': precision_score(y_test_before, pred_before, zero_division=0),
    'recall': recall_score(y_test_before, pred_before, zero_division=0),
    'f1': f1_score(y_test_before, pred_before, zero_division=0),
    'roc_auc': roc_auc_score(y_test_before, prob_before),
    'average_precision': average_precision_score(y_test_before, prob_before),
    'precision_at_20': precision_at_k(y_test_before, prob_before, 20),
    'precision_at_50': precision_at_k(y_test_before, prob_before, 50),
    'precision_at_100': precision_at_k(y_test_before, prob_before, 100),
}

for key, value in metrics_before.items():
    print(f'  {key}: {value:.4f}')

In [ ]:
# AFTER: grouped client-holdout split
print('\n=== AFTER: grouped client-holdout split ===')
unique_clients = frame['client_id'].fillna('unknown').astype(str).unique()
rng = np.random.default_rng(42)
shuffled_unique = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_unique) * 0.2)))
test_clients_after = set(shuffled_unique[:test_client_count])
test_mask_after = frame['client_id'].fillna('unknown').astype(str).isin(test_clients_after).to_numpy()
train_idx_after = np.arange(len(frame))[~test_mask_after]
test_idx_after = np.arange(len(frame))[test_mask_after]

X_train_after = X.iloc[train_idx_after]
X_test_after = X.iloc[test_idx_after]
y_train_after = y.iloc[train_idx_after]
y_test_after = y.iloc[test_idx_after]

model_after = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])
model_after.fit(X_train_after, y_train_after)
prob_after = model_after.predict_proba(X_test_after)[:, 1]
pred_after = (prob_after >= 0.5).astype(int)

metrics_after = {
    'accuracy': accuracy_score(y_test_after, pred_after),
    'precision': precision_score(y_test_after, pred_after, zero_division=0),
    'recall': recall_score(y_test_after, pred_after, zero_division=0),
    'f1': f1_score(y_test_after, pred_after, zero_division=0),
    'roc_auc': roc_auc_score(y_test_after, prob_after),
    'average_precision': average_precision_score(y_test_after, prob_after),
    'precision_at_20': precision_at_k(y_test_after, prob_after, 20),
    'precision_at_50': precision_at_k(y_test_after, prob_after, 50),
    'precision_at_100': precision_at_k(y_test_after, prob_after, 100),
}

for key, value in metrics_after.items():
    print(f'  {key}: {value:.4f}')

In [ ]:
comparison = pd.DataFrame({
    'metric': list(metrics_before.keys()),
    'random_split': list(metrics_before.values()),
    'grouped_split': list(metrics_after.values()),
})
comparison['difference'] = comparison['grouped_split'] - comparison['random_split']

print('\n=== BEFORE / AFTER COMPARISON ===')
print(comparison.round(4).to_string(index=False))

print('\nInterpretation:')
print('A large drop from random split to grouped split suggests the model may be relying on repeated client patterns or weakly separated examples.')
print('A smaller drop suggests the model is more robust under honest validation.')
print('Either way, the grouped split is the more relevant estimate for real-world decision-support.')

## 3) Leakage audit

The paper is a reminder that methodology matters. I now audit the same risks in my own model.

Specific things I am checking:
1. Is the target or a target-derived field being used as a feature?
2. Is the feature window overlapping the target window?
3. Are client or page patterns duplicated across train and test?
4. Are product flags or decision outputs being fed back in as input?

In [ ]:
print('=== Leakage audit ===')
print('1. Target leakage check:')
print('   - trend_direction is used to define the target label, and is not used as a feature.')
print('   - trend_pct is also a target-derived signal and should not be treated as a general feature.')
print('')
print('2. Feature-window overlap:')
print('   - The model uses recent observable signals like impressions, CTR, engagement, and position.')
print('   - These should be treated as decision-support signals, not as a forecast of future causal outcomes.')
print('')
print('3. Client leakage:')
print('   - The grouped split holds out entire clients. This is a more honest validation design than a random row split.')
print('')
print('4. Product-flag leakage:')
print('   - Product rule outputs, when present, should not be used as ordinary model features unless explicitly approved and audited.')
print('')
print('Overall: the model is not a causal statement about what caused the decline. It is a ranking method for prioritizing review candidates from measurable signals.')

## 4) Real failure examples

The paper's lesson is not that model building is useless. It is that a model becomes more credible when we inspect the mistakes and ask what kind of pages the model is getting wrong.

I will examine false positives and false negatives under the grouped validation split, because that is closer to what matters in practice.

In [ ]:
# Build the grouped-test error table
error_frame = frame.iloc[test_idx_after].copy()
error_frame['y_true'] = y_test_after.to_numpy()
error_frame['probability'] = prob_after
error_frame['prediction'] = (prob_after >= 0.5).astype(int)
error_frame['is_fp'] = (error_frame['prediction'] == 1) & (error_frame['y_true'] == 0)
error_frame['is_fn'] = (error_frame['prediction'] == 0) & (error_frame['y_true'] == 1)

false_positives = error_frame[error_frame['is_fp']].sort_values('probability', ascending=False).head(10)
false_negatives = error_frame[error_frame['is_fn']].sort_values('probability', ascending=True).head(10)

print('False positives (model high probability, label not declining):')
if false_positives.empty:
    print('No false positives in this split.')
else:
    display_cols = ['content_id', 'client_id', 'probability', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'days_since_last_update']
    print(false_positives[display_cols].to_string(index=False))

print('\nFalse negatives (label declining, model low probability):')
if false_negatives.empty:
    print('No false negatives in this split.')
else:
    print(false_negatives[display_cols].to_string(index=False))

## 5) Claim rewrite

The paper taught me to be explicit about what is proven and what is only directional.

I will replace my stronger claims with careful, public-safe language.

### Unsafe claim:
"This model identifies the pages most likely to decline and outperforms the baseline by a large margin."

### Safer rewrite:
"On the selected test split, the model ranks pages by measured decline-related signals and produces a higher Precision@K than the baseline rule on that same validation set. This is directional evidence that the learned ranking can prioritize review candidates better than a fixed heuristic on this slice. It is best interpreted as a decision-support tool for manual review, not a causal statement about which pages will definitely decline or recover."

### Why this rewrite is better:
- It names the metric and the split.
- It avoids guaranteed language.
- It makes clear this is not causal proof.
- It keeps the result useful without overstating it.

## 6) Self-check

Before submitting, I confirm that I have:

- [ ] Read the paper and used its main lessons: position matters, and honest validation matters.
- [ ] Picked two findings from the paper and framed a methodology question for each.
- [ ] Re-run my Week 5 model on a grouped or time-aware split.
- [ ] Compared the random-split result to the honest-split result.
- [ ] Audited leakage risks for target overlap, client separation, and feature definitions.
- [ ] Inspected false positives and false negatives and asked what they mean.
- [ ] Rewritten my strongest claims using measured, directional, and decision-support language.
- [ ] Avoided causality or certainty claims that the data cannot support.
- [ ] Saved the notebook under `work/notebooks/w06_validation_audit.ipynb`.